# Police Precinct Cleaning & Assignment

**Goals:**
1. Convert LAPD station projected coordinates (EPSG:2229) → WGS84 lat/lon
2. Merge LAPD station + division data → `data/LAPD/lapd_precincts_combined.csv`
3. Assign LA County encampment sites to nearest precinct via KDTree (Voronoi-equivalent nearest-neighbor)
4. Confirm MyLA311 data already has `PolicePrecinct` populated

In [1]:
# Cell 1: Imports
from pathlib import Path

import numpy as np
import pandas as pd
from pyproj import Transformer
from scipy.spatial import KDTree

DATA = Path("data")

In [2]:
# Cell 2: Load LAPD Police Stations and convert projected coords to WGS84
#
# The station file already has x,y in CA State Plane Zone 5 (EPSG:2229, US Survey Feet).
# We reproject to WGS84 (EPSG:4326) to get standard lat/lon — more accurate than geocoding.
# always_xy=True ensures the transformer returns (lon, lat) in that order.

stations = pd.read_csv(DATA / "LAPD" / "LAPD_Police_Stations_9134071724237280264.csv")

transformer = Transformer.from_crs("EPSG:2229", "EPSG:4326", always_xy=True)
stations["lon"], stations["lat"] = transformer.transform(
    stations["x"].values, stations["y"].values
)
stations = stations.drop(columns=["x", "y"])

print(stations[["DIVISION", "PREC", "LOCATION", "lat", "lon"]].to_string(index=False))

        DIVISION  PREC                          LOCATION       lat         lon
          HARBOR     5         2175 JOHN S. GIBSON BLVD. 33.757656 -118.289229
       SOUTHEAST    18                  145 W. 108TH ST. 33.938623 -118.275382
     77TH STREET    12                  7600 S. BROADWAY 33.970303 -118.277657
         PACIFIC    14                12312 CULVER BLVD. 33.991650 -118.419829
       SOUTHWEST     3 1546 MARTIN LUTHER KING JR. BLVD. 34.010570 -118.305129
          NEWTON    13              3400 S. CENTRAL AVE. 34.012351 -118.256107
         CENTRAL     1                    251 E. 6TH ST. 34.044015 -118.247282
WEST LOS ANGELES     8                  1663 BUTLER AVE. 34.043773 -118.450767
      HOLLENBECK     4                   2111 E. 1ST ST. 34.045004 -118.213056
        WILSHIRE     7                 4861 VENICE BLVD. 34.046743 -118.342817
         OLYMPIC    20              1130 S. VERMONT AVE. 34.050204 -118.291164
         RAMPART     2                   1401 W. 6TH

In [3]:
# Cell 3: Merge stations with division data and save combined CSV
#
# LAPD_Division has precinct-level area/perimeter stats (no polygon geometry).
# We join on PREC so each station row also carries its division's area metadata.

divisions = pd.read_csv(DATA / "LAPD" / "LAPD_Division_109032480992066415.csv")

# OBJECTID exists in both — disambiguate before merging
divisions = divisions.rename(columns={"OBJECTID": "DIV_OBJECTID"})

combined = stations.merge(divisions, on="PREC", how="left")

out_path = DATA / "LAPD" / "lapd_precincts_combined.csv"
combined.to_csv(out_path, index=False)
print(f"Saved {len(combined)} rows to {out_path}")
combined.head()

Saved 21 rows to data/lapd_precincts_combined.csv


,OBJECTID,DIVISION,LOCATION,PREC,lon,lat,DIV_OBJECTID,APREC,AREA,PERIMETER,Shape__Area,Shape__Length
0,1,HARBOR,2175 JOHN S. GIBSON BLVD.,5,-118.289229,33.757656,20,HARBOR,8.928780e+08,272451.139908,8.928780e+08,272451.172888
1,2,SOUTHEAST,145 W. 108TH ST.,18,-118.275382,33.938623,19,SOUTHEAST,2.611391e+08,111470.973394,2.611391e+08,111470.973394
2,3,77TH STREET,7600 S. BROADWAY,12,-118.277657,33.970303,18,77TH STREET,3.159590e+08,114137.816270,3.159590e+08,114137.816270
3,4,PACIFIC,12312 CULVER BLVD.,14,-118.419829,33.991650,17,PACIFIC,7.176129e+08,246934.321606,7.176129e+08,246934.321691
4,5,SOUTHWEST,1546 MARTIN LUTHER KING JR. BLVD.,3,-118.305129,34.010570,15,SOUTHWEST,3.433255e+08,102632.208443,3.433255e+08,102632.208443


In [4]:
# Cell 4: Assign LA County encampment sites to police precincts via nearest-neighbor
#
# Method: KDTree on station (lat, lon) coordinates. For each county site, the nearest
# station is its assigned precinct — equivalent to Voronoi cell membership.
# Sites on a cell boundary are handled deterministically by KDTree (no ambiguity
# with floating-point coords), satisfying the "assign by closest" tiebreak rule.
#
# Note: KDTree uses Euclidean distance in degree-space. Over the small extent of
# LA County (~1° lat × ~1.5° lon), this is accurate enough for precinct-level assignment.

county = pd.read_csv(DATA / "LAHSA" / "LA_County_Homeless_Encampment_Request_Forms.csv")
county_clean = county.dropna(subset=["X", "Y"]).copy()
print(f"{len(county)} total rows, {len(county_clean)} with valid coordinates")

# Build tree from station (lat, lon); query with site (Y=lat, X=lon)
station_coords = combined[["lat", "lon"]].values
tree = KDTree(station_coords)

site_coords = county_clean[["Y", "X"]].values  # Y=lat, X=lon in this dataset
_, indices = tree.query(site_coords)

county_clean["PolicePrecinct"] = combined["DIVISION"].values[indices]

out_path = DATA / "LAHSA" / "LA_County_Homeless_Encampment_Request_Forms_with_precinct.csv"
county_clean.to_csv(out_path, index=False)
print(f"\nSaved {len(county_clean)} rows to {out_path}")
print("\nPrecinct distribution:")
print(county_clean["PolicePrecinct"].value_counts())

898 total rows, 898 with valid coordinates

Saved 898 rows to data/LA_County_Homeless_Encampment_Request_Forms_with_precinct.csv

Precinct distribution:
PolicePrecinct
HOLLENBECK          447
SOUTHEAST           154
77TH STREET          51
HARBOR               46
NORTH HOLLYWOOD      42
VAN NUYS             27
NEWTON               23
PACIFIC              22
WEST VALLEY          19
MISSION              17
TOPANGA              15
DEVONSHIRE           13
FOOTHILL             12
NORTHEAST             5
WEST LOS ANGELES      3
SOUTHWEST             1
WILSHIRE              1
Name: count, dtype: int64


In [5]:
# Cell 5: Normalize and verify MyLA311 PolicePrecinct column
#
# Raw values are a mix of division names ('CENTRAL') and numeric precinct numbers
# ('1.0'). We apply the same PREC→DIVISION lookup used in map_builder.py so the
# notebook reflects exactly what the map displays.

prec_lookup = dict(zip(combined["PREC"].astype(int), combined["DIVISION"]))

def normalize_precinct(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    text = str(val).strip()
    if not text:
        return None
    try:
        n = int(float(text))
        return prec_lookup.get(n) if n != 0 else None
    except ValueError:
        return text

myla = pd.read_csv(
    DATA / "MyLA311" / "MyLA311_Service_Request_Homeless_Encampment_Combined_2025_20260524.csv",
    low_memory=False,
    usecols=["PolicePrecinct"],
)
myla["PolicePrecinct"] = myla["PolicePrecinct"].apply(normalize_precinct)
null_pct = myla["PolicePrecinct"].isna().mean() * 100
print(f"Null/unmapped rate: {null_pct:.1f}%")
print(myla["PolicePrecinct"].value_counts())

Null/unmapped rate: 1.4%
PolicePrecinct
RAMPART             9720
NEWTON              9215
NORTH HOLLYWOOD     7601
OLYMPIC             7563
PACIFIC             6999
WEST LOS ANGELES    6104
TOPANGA             4454
WEST VALLEY         3909
HOLLENBECK          3888
HOLLYWOOD           3856
VAN NUYS            3808
NORTHEAST           3797
CENTRAL             3686
MISSION             3607
FOOTHILL            3267
77TH STREET         2859
WILSHIRE            2761
DEVONSHIRE          2594
SOUTHWEST           2333
SOUTHEAST           2220
HARBOR               987
Name: count, dtype: int64


,PolicePrecinct
0,NEWTON
1,WILSHIRE
2,VAN NUYS
3,RAMPART
4,HOLLYWOOD
